In [7]:
import pandas as pd
import plotly.express as px

# 1. Load the data 
# We use nrows=50 to grab only the actual trials and ignore the summary text at the bottom of the CSV
df = pd.read_csv("bo_results_20260724_170359.csv", nrows=50)

# 2. Create the interactive 3D scatter plot
fig = px.scatter_3d(
    df, 
    x='RPM_1', 
    y='RPM_2', 
    z='RPM_3',
    color='Displacement_mm',
    size='Displacement_mm',      # Larger dots = further displacement
    size_max=25,                 # Max dot size
    color_continuous_scale='Inferno', # A great colormap for heat (Black -> Purple -> Orange -> Yellow)
    title='SPVVVECTR Tensegrity BO Results (Random Priors)',
    hover_data=['Trial_Number', 'Phase'],
    labels={
        'RPM_1': 'Strut 1 (RPM)',
        'RPM_2': 'Strut 2 (RPM)',
        'RPM_3': 'Strut 3 (RPM)',
        'Displacement_mm': 'Distance (mm)'
    }
)

# 3. Adjust the layout for better viewing
fig.update_layout(
    scene=dict(
        xaxis=dict(range=[-1000, 1000]),
        yaxis=dict(range=[-1000, 1000]),
        zaxis=dict(range=[-1000, 1000])
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

# 4. Display the graph
fig.show(renderer="browser")

Above is the heatmap from just data points we observed. Below is the surrogate function from bayesian optimization.

In [6]:
import numpy as np
import plotly.graph_objects as go
from skopt import load

# 1. Load the "brain" (The Gaussian Process model)
pkl_filename = "bo_model_20260724_170359.pkl"  # e.g., "Data_Logs/bo_model_20260724_174100.pkl"
res = load(pkl_filename)
gp_model = res.models[-1]  # Grab the final, fully-trained version of the model

# 2. Create a dense 3D grid of untested points
# We use 20 points per axis, which creates a grid of 8,000 unique RPM combinations
axis_vals = np.linspace(-1000, 1000, 20)
R1, R2, R3 = np.meshgrid(axis_vals, axis_vals, axis_vals)
grid_points_raw = np.c_[R1.ravel(), R2.ravel(), R3.ravel()]

# 3. Ask the Gaussian Process to predict the displacement for all 8,000 points
# skopt requires us to transform the raw points into its internal mathematical scale first
grid_points_transformed = res.space.transform(grid_points_raw.tolist())
predicted_displacement = -gp_model.predict(grid_points_transformed) # Negate because skopt minimizes

# 4. Build the 3D Volume (Heatmap)
fig = go.Figure(data=go.Volume(
    x=grid_points_raw[:, 0],
    y=grid_points_raw[:, 1],
    z=grid_points_raw[:, 2],
    value=predicted_displacement,
    isomin=np.min(predicted_displacement),
    isomax=np.max(predicted_displacement),
    opacity=0.2,             # Translucent so we can see inside the cube
    surface_count=15,        # Creates 15 "shells" of heat contour
    colorscale='Inferno',
    colorbar=dict(title="Predicted<br>Displacement (mm)")
))

fig.update_layout(
    scene=dict(
        xaxis_title='Strut 1 (RPM)',
        yaxis_title='Strut 2 (RPM)',
        zaxis_title='Strut 3 (RPM)'
    ),
    title="Gaussian Process Surrogate Model: Predicted Displacement Landscape",
    margin=dict(l=0, r=0, b=0, t=40)
)

# Open in browser for lag-free 3D rotation
fig.show(renderer="browser")